In [ ]:
# AIM: CNN with performance improvement + comparison graphs

import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

# 1. Load dataset
data = load_digits()
x = data.images
y = data.target

# 2. Preprocessing
x = x.reshape(-1, 8, 8, 1)
x = x / 16.0

# 3. Train-test split
x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=42
)

# 4. Create TWO models (for comparison)

# --- Model 1: Basic (no tuning) ---
def create_basic_model():
    model = models.Sequential([
        layers.Conv2D(32, (3,3), activation='relu', input_shape=(8,8,1)),
        layers.Flatten(),
        layers.Dense(64, activation='relu'),
        layers.Dense(10, activation='softmax')
    ])
    
    model.compile(optimizer='adam',
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
    return model


# --- Model 2: Improved (with tuning) ---
def create_improved_model():
    model = models.Sequential([
        layers.Conv2D(32, (3,3), activation='relu', input_shape=(8,8,1)),
        layers.MaxPooling2D((2,2)),
        
        layers.Conv2D(64, (3,3), activation='relu'),
        layers.Flatten(),
        
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.5),  # prevents overfitting
        
        layers.Dense(10, activation='softmax')
    ])
    
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
    return model


# 5. Train both models

basic_model = create_basic_model()
improved_model = create_improved_model()

print("Training Basic Model...")
history_basic = basic_model.fit(
    x_train, y_train,
    epochs=10,
    validation_data=(x_test, y_test),
    verbose=1
)

print("\nTraining Improved Model...")
history_improved = improved_model.fit(
    x_train, y_train,
    epochs=10,
    validation_data=(x_test, y_test),
    verbose=1
)


# 6. Evaluate
basic_loss, basic_acc = basic_model.evaluate(x_test, y_test)
improved_loss, improved_acc = improved_model.evaluate(x_test, y_test)

print("\nBasic Model Accuracy:", basic_acc)
print("Improved Model Accuracy:", improved_acc)


# 7. Plot comparison graphs

plt.figure(figsize=(12,5))

# Accuracy graph
plt.subplot(1,2,1)
plt.plot(history_basic.history['accuracy'], label='Basic Train Acc')
plt.plot(history_basic.history['val_accuracy'], label='Basic Val Acc')

plt.plot(history_improved.history['accuracy'], label='Improved Train Acc')
plt.plot(history_improved.history['val_accuracy'], label='Improved Val Acc')

plt.title("Accuracy Comparison")
plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.legend()


# Loss graph
plt.subplot(1,2,2)
plt.plot(history_basic.history['loss'], label='Basic Train Loss')
plt.plot(history_basic.history['val_loss'], label='Basic Val Loss')

plt.plot(history_improved.history['loss'], label='Improved Train Loss')
plt.plot(history_improved.history['val_loss'], label='Improved Val Loss')

plt.title("Loss Comparison")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.legend()

plt.show()